# **DATA PROCESSING**

In [1]:
import sys

!pip install facenet-pytorch

cfg_path = "/content/drive/MyDrive/11/src/"
if cfg_path not in sys.path:
  sys.path.append(cfg_path)

import os
import torch
from facenet_pytorch import MTCNN
from PIL import Image, ImageOps
from tqdm import tqdm
import numpy as np
import pandas as pd
from config import *
import re
from pathlib import Path

## **Pipeline di Preprocessing e Allineamento**

Questa sezione definisce la classe `FacePipeline`, progettata per automatizzare la preparazione del dataset grezzo. Le operazioni principali includono:

1.  **Rilevamento e Allineamento (MTCNN):**
    * Ogni immagine viene analizzata per individuare il volto.
    * Viene calcolato l'angolo di rotazione basato sulla posizione degli occhi (landmarks) per allineare il volto orizzontalmente.
    * Il volto allineato viene ritagliato e salvato.
2.  **Generazione Dataset Strutturato:**
    * La pipeline itera sulle cartelle (split `train`/`val`).
    * Genera automaticamente un file **CSV (manifest)** contenente i percorsi delle immagini processate e le relative label numeriche, essenziale per il caricamento efficiente tramite `DataLoader`.

In [6]:
class FacePipeline:
  """
  Gestisce l'intera pipeline di preprocessing: rilevamento volti (MTCNN),
  allineamento geometrico basato sugli occhi e generazione dei manifest CSV.
  """
  def __init__(self, base_input_root, base_output_root):
    """
    Configura i percorsi di input/output e inizializza il modello MTCNN
    sul device disponibile (GPU o CPU) con parametri ottimizzati.
    """
    self.base_input_root = base_input_root
    self.base_output_root = base_output_root

    self.device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
    print(f"Pipeline inizializzata su device: {self.device}")

    self.mtcnn = MTCNN(
        keep_all=False,
        select_largest=True,
        margin=20,
        post_process=False,
        device=self.device
    )

  def _extract_label(self, subject_name):
    """
    Estrae l'etichetta numerica (int) dal nome della cartella del soggetto (es. 'id_001' -> 1).
    Restituisce -1 se nessun numero viene trovato.
    """
    match = re.search(r'\d+', subject_name)
    if match:
        return int(match.group())
    else:
        print(f"Attenzione: Nessun numero trovato in '{subject_name}'. Assegno label provvisoria -1.")
        return -1

  def _align_and_save(self, img_path, save_full_path):
    """
    Processa la singola immagine: rileva il volto, calcola l'angolo di rotazione
    basato sugli occhi, allinea l'immagine e salva il crop finale.
    Restituisce True se l'operazione ha successo.
    """
    try:
        img = Image.open(img_path)
        img = ImageOps.exif_transpose(img)
        img = img.convert('RGB')

        boxes, probs, landmarks = self.mtcnn.detect(img, landmarks=True)

        if boxes is None or landmarks is None:
            return False

        left_eye = landmarks[0][0]
        right_eye = landmarks[0][1]
        delta_x = right_eye[0] - left_eye[0]
        delta_y = right_eye[1] - left_eye[1]
        angle = np.degrees(np.arctan2(delta_y, delta_x))

        if abs(angle) > 1:
            img_rotated = img.rotate(angle, resample=Image.BICUBIC, expand=True)
            self.mtcnn(img_rotated, save_path=save_full_path)
        else:
            self.mtcnn(img, save_path=save_full_path)
        return True
    except Exception as e:
        return False

  def process_split(self, split_name):
    """
    Itera su una specifica partizione del dataset (es. 'train' o 'val'), esegue
    l'allineamento per tutte le immagini e genera il file CSV riassuntivo.
    """
    input_split_path = os.path.join(self.base_input_root, split_name)
    output_split_path = os.path.join(self.base_output_root, split_name)

    if not os.path.exists(input_split_path):
        print(f"ERRORE: La cartella '{split_name}' non esiste in {self.base_input_root}")
        return

    print(f"\n--- Inizio elaborazione: {split_name.upper()} ---")

    dataset_records = []

    subjects = sorted([d for d in os.listdir(input_split_path) if os.path.isdir(os.path.join(input_split_path, d))])

    for subject_name in tqdm(subjects, desc=f"Soggetti in {split_name}"):

        label = self._extract_label(subject_name)

        subject_input_dir = os.path.join(input_split_path, subject_name)
        subject_output_dir = os.path.join(output_split_path, subject_name)
        os.makedirs(subject_output_dir, exist_ok=True)

        for root, _, files in os.walk(subject_input_dir):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):

                    file_path = os.path.join(root, file)
                    parent_name = os.path.basename(root)

                    unique_filename = f"{file}"
                    save_full_path = os.path.join(subject_output_dir, unique_filename)

                    success = self._align_and_save(file_path, save_full_path)

                    if success:
                        dataset_records.append({
                            'image_path': save_full_path,
                            'label': label,
                            'subject_name': subject_name,
                            'split': split_name
                        })

    if dataset_records:
        df = pd.DataFrame(dataset_records)
        csv_name = f"manifest_{split_name}.csv"
        csv_path = os.path.join(self.base_output_root, csv_name)
        df.to_csv(csv_path, index=False)
        print(f"Salvato manifest per {split_name} in: {csv_path}")
    else:
        print(f"Nessuna immagine valida trovata in {split_name}")

In [7]:
DATASET_ROOT = "/content/drive/MyDrive/soggetto"
DATASET_PROCESSED_ROOT = "/content/drive/MyDrive/subject_19"
pipeline = FacePipeline(DATASET_ROOT, DATASET_PROCESSED_ROOT)

pipeline.process_split("galleries")
pipeline.process_split("probes")

Pipeline inizializzata su device: cpu

--- Inizio elaborazione: GALLERIES ---


Soggetti in galleries: 100%|██████████| 1/1 [00:56<00:00, 56.97s/it]


Salvato manifest per galleries in: /content/drive/MyDrive/subject_19/manifest_galleries.csv

--- Inizio elaborazione: PROBES ---


Soggetti in probes: 100%|██████████| 1/1 [02:06<00:00, 126.65s/it]

Salvato manifest per probes in: /content/drive/MyDrive/subject_19/manifest_probes.csv


## Generazione Mappa Dataset (CSV)

Questa funzione esegue la scansione della directory radice per creare un **manifest CSV** strutturato, essenziale per la gestione dei dati nel training.

* **Logica di Scansione:** Il codice naviga attraverso gli split standard (`train`, `test`, `val`) e le classi target (`live`, `spoof`).
* **Output:** Genera un file `dataset_labels.csv` che mappa ogni file immagine ai suoi metadati:
    * `filename`: Nome del file.
    * `split`: La partizione del dataset.
    * `label`: L'etichetta di classe (`live` o `spoof`).

In [ ]:
def create_dataset_csv(root_folder, output_csv_name="dataset_map.csv"):
    """
    Scansiona la struttura delle cartelle e genera un CSV.

    Args:
        root_folder (str): Il percorso della cartella principale.
        output_csv_name (str): Il nome del file CSV di output.
    """

    root_path = Path(root_folder)

    splits = ['train', 'test', 'val']
    labels = ['live', 'spoof']

    data = []

    valid_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff'}

    print(f"Scansione iniziata in: {root_path}...")

    for split in splits:
        for label in labels:
            target_dir = root_path / split / label

            if target_dir.exists():
                for file_path in target_dir.iterdir():
                    if file_path.is_file() and file_path.suffix.lower() in valid_extensions:
                        data.append({
                            'filename': file_path.name,
                            'split': split,
                            'label': label
                        })
            else:
                print(f"Attenzione: La cartella '{target_dir}' non esiste e sarà saltata.")

    df = pd.DataFrame(data)

    if not df.empty:
        df = df.sort_values(by=['split', 'label'])

        df.to_csv(output_csv_name, index=False)
        print(f"--- Fatto! ---")
        print(f"File salvato come: {output_csv_name}")
        print(f"Totale immagini trovate: {len(df)}")
    else:
        print("Errore: Nessuna immagine trovata. Controlla il percorso della cartella.")

DATASET_ANTISPOOF_FINAL = "/content/drive/MyDrive/Spoof_finale"
create_dataset_csv(DATASET_ANTISPOOF_FINAL, "dataset_labels.csv")

Scansione iniziata in: /content/drive/MyDrive/Spoof_finale...
--- Fatto! ---
File salvato come: dataset_labels.csv
Totale immagini trovate: 983
